입력: prices (정수배열) => 하루마다의 주식 가격
출력: 매수 매도를 통한 최대 이익의 값

한번에 한주까지만 보유 가능하지만 같은날, 여러번 매도 매수가능
양의 이익이 안나오면 안사는게 최대 이익

마지막 날에 최솟값이 있는거 아니면 일단 무조건 매수 하면 이익이 나오긴 함


어떤 규칙을 적용해야 최대 이익이 나오는지 모르겠음

언제 매수할지 부터 규칙을 정한다면?...
일단 최댓값으론 절대 사면안됨
마지막 날에 있지 않으면 최솟값으로 일단 구매
마지막날 전에 팔았다면, 두번째로 작은 값에 다시 구매

언제 판매할지 규칙을 정한다면?
일단 팔아서 음수가 나오면 안됨
마냥 최대 이익이 나온다고 팔게 아닌게 뒤에 또 사고 팔면 더 이득을 보는 경우도 잇음

[7,1,5,3,6,4]
[1,3,4,5,6,7]

In [ ]:
class Solution:
    def maxProfit(self, prices: List[int]) -> int:
        
        # 우상향이란 전제하에 1,4,7,8 이라면 
        # day 1 : 1에사고 
        # day 2 : 4에팔고 4에사서(같은날에 매도매수 여러번 가능) 
        # day 3 : 7팔고 7에 사서 
        # day 4 : 8에 팔아 번 최종 이득인 7이

        # day 1 : 1에사서 
        # day 4 : 8에 팔아 번 이득 7과 같음

        # 즉 prices 배열로 내일 가격을 알수 dlT으므로 우상향 한다면 무조건 매수 매도를 통해 이득을 누적시키고
        # 내일 떨어진다면 팔지말고 들고 있어야함 
        
        # => 오르는 구간의 차이를 전부 더하면 최대 이익 
        profit = 0
        for i in range(0,len(prices)-1):
            if prices[i] < prices[i+1]:
                profit += (prices[i+1]-prices[i])
        return profit

# 분류 : C

좋아. 네가 남긴 고민 흔적을 보면, **문제를 아무 생각 없이 찍은 게 아니라 꽤 정상적인 방향으로 고민하다가 “규칙을 잘못 세운” 상태**야.

핵심 진단부터 말하면:
좋아. 네가 남긴 고민 흔적을 보면, **문제를 아무 생각 없이 찍은 게 아니라 꽤 정상적인 방향으로 고민하다가 “규칙을 잘못 세운” 상태**야.

핵심 진단부터 말하면:

> 이 문제는 **전역적인 최솟값/최댓값을 찾는 문제처럼 보이지만**,
> 실제로는 **매일의 국소적인 상승분을 모두 챙기는 그리디 문제**야.

---

## 1. 유형

정답 유형은 **그리디**야.

엄밀히는 DP로도 풀 수 있어.
예를 들면:

* `cash`: 오늘까지 주식을 안 들고 있을 때 최대 이익
* `hold`: 오늘까지 주식을 들고 있을 때 최대 이익

이런 식으로 상태를 둘 수 있어.

하지만 이 문제는 조건이 단순해:

* 거래 횟수 제한 없음
* 수수료 없음
* 쿨다운 없음
* 한 번에 한 주만 보유

이 조건에서는 DP를 끝까지 밀 필요 없이,
**오르는 구간의 차이를 전부 더하면 최대 이익**이 돼서 그리디가 정답이야.

즉 분류는 이렇게 보면 돼:

* **BFS:** 아님
* **투포인터:** 아님
* **DP:** 가능은 하지만 과함
* **그리디:** 정답

---

## 2. 네가 막힌 이유

네 막힘은 **구현 실수**가 아니라 **문제 해석 프레임**에서 왔어.

### 네가 세운 프레임

너는 계속 이런 식으로 생각했어:

* 언제 사지?
* 언제 팔지?
* 최솟값에서 사야 하나?
* 최댓값에서 팔아야 하나?
* 한 번 팔면 다음엔 두 번째 최솟값을 찾아야 하나?

이건 **“큰 거래 몇 번으로 최적해를 만든다”**는 프레임이야.

그런데 이 문제는 그렇게 안 봐야 해.

### 왜 그 프레임이 막히냐

이 문제는 **최적의 매수/매도 시점 몇 개를 뽑는 문제**가 아니라,
**오르는 날들의 이익을 빠짐없이 수집하는 문제**야.

예를 들어:

```python
prices = [1, 3, 2, 4]
```

네 프레임대로면:

* 1에 사서 4에 판다 → 이익 3

그런데 실제 최대는:

* 1 → 3 : +2
* 2 → 4 : +2
* 총 4

즉,
**전역 최솟값에서 사서 전역 최댓값에 파는 사고방식이 최대 이익을 놓칠 수 있어.**

왜냐하면 중간에 한 번 내려갔다가 다시 오르면,
그 하강 직전에 팔고 다시 사는 게 더 이득이기 때문이야.

---

## 3. 네 사고에서 좋았던 점 / 틀어진 점

### 좋았던 점

이건 잘 봤어:

* **최댓값에서 사면 안 된다**
* **음수 이익이면 안 산다**
* **한 번 판 뒤 다시 사는 경우가 있다**
* **한 번의 큰 이익만 보면 안 될 수도 있다**

이건 문제의 본질 근처까지 간 거야.

### 틀어진 점

틀어진 지점은 여기야:

* “최솟값을 찾고 산다”
* “최댓값을 찾고 판다”
* “두 번째로 작은 값에 다시 산다”

이건 **정렬/전역 비교 관점**인데,
이 문제는 사실 **인접한 날짜끼리 비교**하면 끝나.

즉 질문이 이렇게 바뀌어야 해:

* “오늘보다 내일이 더 비싼가?”
* 더 비싸면 그 상승분은 챙겨라

이걸 끝까지 반복하면 돼.

---

## 4. 진짜 규칙

이 문제의 규칙은 딱 하나야.

> **내일 가격이 오늘 가격보다 높으면, 그 차이만큼 이익에 더한다.**

코드로는:

```python
profit += max(0, prices[i] - prices[i - 1])
```

왜 이게 되냐면,
상승 구간 전체의 이익은 그 사이의 **하루하루 오른 값들의 합**과 같기 때문이야.

예를 들어:

```python
[1, 2, 3, 4]
```

* 1에 사서 4에 팔기 = 3
* (2-1) + (3-2) + (4-3) = 1 + 1 + 1 = 3

완전히 같아.

그러니까 굳이 “어디서 사고 어디서 팔지”를 멀리 내다볼 필요가 없어.
**오른 날만 먹으면 된다.**

---

## 5. 막힌 이유 분류

네가 요청한 포맷대로 분류하면 이래.

### 유형

**그리디**

### 막힌 이유

가장 정확한 표현은:

**상태 정의 못함**보다는
**문제를 전역 최적 매수/매도 시점 탐색으로 해석해서 규칙 설계가 꼬임**

조금 더 쪼개면:

* 시간복잡도 계산 실패: 핵심 아님
* 상태정의 못함: DP 관점에서는 해당될 수 있음
* 구현 실수: 아직 구현 단계까지 가기 전
* **진짜 원인:** 문제를 보는 틀 자체가 잘못 잡힘

즉,
**“최솟값/최댓값 기반의 큰 거래 설계”로 접근해서 막힘**

---

## 6. 어떤 신호에서 이 풀이를 떠올려야 하나

앞으로 아래 신호가 보이면 바로 이 문제를 떠올리면 돼.

### 신호 1

**여러 번 거래 가능**

→ 한 번의 매수/매도로 끝내는 문제가 아닐 수 있음

### 신호 2

**동시에 여러 주식은 못 듦**

→ 매수/매도 순서만 지키면 됨
→ 거래를 쪼개는 것이 가능함

### 신호 3

**수수료 없음 / 쿨다운 없음 / 거래 횟수 제한 없음**

→ 오르는 구간을 전부 챙겨도 손해가 없음
→ 그리디 가능성 매우 큼

### 신호 4

**가격 배열이 시간 순서대로 주어짐**

→ 정렬해서 풀면 안 되고, 인접 관계가 중요함

이 신호들을 보면 바로:

> “아, 오르는 날의 차이를 다 더하는 문제구나”

를 떠올리면 돼.

---

## 7. 파이썬 포인트

이 문제에서 중요한 파이썬 포인트는 많지 않아.
자료구조 문제라기보다 **배열 순회** 문제라서 그래.

### 꼭 필요한 것

* `range(1, len(prices))`
* 인접 원소 비교
* 누적합 변수

예시:

```python
from typing import List

class Solution:
    def maxProfit(self, prices: List[int]) -> int:
        profit = 0

        for i in range(1, len(prices)):
            if prices[i] > prices[i - 1]:
                profit += prices[i] - prices[i - 1]

        return profit
```

### 있으면 깔끔한 것

`sum` + generator

```python
class Solution:
    def maxProfit(self, prices: List[int]) -> int:
        return sum(
            max(0, prices[i] - prices[i - 1])
            for i in range(1, len(prices))
        )
```

### 안 필요한 것

이 문제에선 거의 필요 없음:

* `deque`
* `set`
* `defaultdict`
* `sort key`

`enumerate`도 쓸 수는 있지만 필수는 아니야.

---

## 8. 네 접근이 왜 막혔는지 한 문장으로 요약

> **“언제 한 번 크게 먹을까?”를 찾으려 해서 막혔고,
> 사실은 “오를 때마다 조금씩 다 먹는다”가 정답이었다.**

---

## 9. 앞으로 같은 류에서 떠올릴 문장

이 문장을 머리에 박아두면 좋아.

> **거래 제한 없고 수수료/쿨다운 없으면, 양수인 인접 차이를 전부 더하자.**

이게 거의 트리거 문장이다.

---

## 10. 네 메모를 더 좋은 형태로 바꾸면

네 사고 흔적을 문제 풀이용 언어로 바꾸면 이렇게 정리할 수 있어:

* 최솟값/최댓값을 찾는 문제가 아니다.
* 상승 구간이 여러 번 나오면 거래도 여러 번 해야 한다.
* 전체 최대 이익은 각 상승분의 합과 같다.
* 따라서 `prices[i] > prices[i-1]`일 때만 차이를 더하는 그리디를 쓴다.

---

## 11. 이 문제에서 네가 얻어가야 할 진짜 교훈

이 문제의 교훈은 주식 문제가 아니라 이것이야:

> **최적화 문제라고 해서 항상 “최적의 한 지점”을 찾아야 하는 건 아니다.**
> 어떤 문제는 **좋은 국소 선택들을 전부 합치면 전역 최적**이 된다.

이게 그리디 감각이야.

원하면 다음 답변에서 네 고민 흔적을 기준으로
`왜 Stock I는 최솟값 추적이고, Stock II는 양수 차이 합인지`까지 비교해서 정리해줄게.

출처: From training data.

> 이 문제는 **전역적인 최솟값/최댓값을 찾는 문제처럼 보이지만**,
> 실제로는 **매일의 국소적인 상승분을 모두 챙기는 그리디 문제**야.

---

## 1. 유형

정답 유형은 **그리디**야.

엄밀히는 DP로도 풀 수 있어.
예를 들면:

* `cash`: 오늘까지 주식을 안 들고 있을 때 최대 이익
* `hold`: 오늘까지 주식을 들고 있을 때 최대 이익

이런 식으로 상태를 둘 수 있어.

하지만 이 문제는 조건이 단순해:

* 거래 횟수 제한 없음
* 수수료 없음
* 쿨다운 없음
* 한 번에 한 주만 보유

이 조건에서는 DP를 끝까지 밀 필요 없이,
**오르는 구간의 차이를 전부 더하면 최대 이익**이 돼서 그리디가 정답이야.

즉 분류는 이렇게 보면 돼:

* **BFS:** 아님
* **투포인터:** 아님
* **DP:** 가능은 하지만 과함
* **그리디:** 정답

---

## 2. 네가 막힌 이유

네 막힘은 **구현 실수**가 아니라 **문제 해석 프레임**에서 왔어.

### 네가 세운 프레임

너는 계속 이런 식으로 생각했어:

* 언제 사지?
* 언제 팔지?
* 최솟값에서 사야 하나?
* 최댓값에서 팔아야 하나?
* 한 번 팔면 다음엔 두 번째 최솟값을 찾아야 하나?

이건 **“큰 거래 몇 번으로 최적해를 만든다”**는 프레임이야.

그런데 이 문제는 그렇게 안 봐야 해.

### 왜 그 프레임이 막히냐

이 문제는 **최적의 매수/매도 시점 몇 개를 뽑는 문제**가 아니라,
**오르는 날들의 이익을 빠짐없이 수집하는 문제**야.

예를 들어:

```python
prices = [1, 3, 2, 4]
```

네 프레임대로면:

* 1에 사서 4에 판다 → 이익 3

그런데 실제 최대는:

* 1 → 3 : +2
* 2 → 4 : +2
* 총 4

즉,
**전역 최솟값에서 사서 전역 최댓값에 파는 사고방식이 최대 이익을 놓칠 수 있어.**

왜냐하면 중간에 한 번 내려갔다가 다시 오르면,
그 하강 직전에 팔고 다시 사는 게 더 이득이기 때문이야.

---

## 3. 네 사고에서 좋았던 점 / 틀어진 점

### 좋았던 점

이건 잘 봤어:

* **최댓값에서 사면 안 된다**
* **음수 이익이면 안 산다**
* **한 번 판 뒤 다시 사는 경우가 있다**
* **한 번의 큰 이익만 보면 안 될 수도 있다**

이건 문제의 본질 근처까지 간 거야.

### 틀어진 점

틀어진 지점은 여기야:

* “최솟값을 찾고 산다”
* “최댓값을 찾고 판다”
* “두 번째로 작은 값에 다시 산다”

이건 **정렬/전역 비교 관점**인데,
이 문제는 사실 **인접한 날짜끼리 비교**하면 끝나.

즉 질문이 이렇게 바뀌어야 해:

* “오늘보다 내일이 더 비싼가?”
* 더 비싸면 그 상승분은 챙겨라

이걸 끝까지 반복하면 돼.

---

## 4. 진짜 규칙

이 문제의 규칙은 딱 하나야.

> **내일 가격이 오늘 가격보다 높으면, 그 차이만큼 이익에 더한다.**

코드로는:

```python
profit += max(0, prices[i] - prices[i - 1])
```

왜 이게 되냐면,
상승 구간 전체의 이익은 그 사이의 **하루하루 오른 값들의 합**과 같기 때문이야.

예를 들어:

```python
[1, 2, 3, 4]
```

* 1에 사서 4에 팔기 = 3
* (2-1) + (3-2) + (4-3) = 1 + 1 + 1 = 3

완전히 같아.

그러니까 굳이 “어디서 사고 어디서 팔지”를 멀리 내다볼 필요가 없어.
**오른 날만 먹으면 된다.**

---

## 5. 막힌 이유 분류

네가 요청한 포맷대로 분류하면 이래.

### 유형

**그리디**

### 막힌 이유

가장 정확한 표현은:

**상태 정의 못함**보다는
**문제를 전역 최적 매수/매도 시점 탐색으로 해석해서 규칙 설계가 꼬임**

조금 더 쪼개면:

* 시간복잡도 계산 실패: 핵심 아님
* 상태정의 못함: DP 관점에서는 해당될 수 있음
* 구현 실수: 아직 구현 단계까지 가기 전
* **진짜 원인:** 문제를 보는 틀 자체가 잘못 잡힘

즉,
**“최솟값/최댓값 기반의 큰 거래 설계”로 접근해서 막힘**

---

## 6. 어떤 신호에서 이 풀이를 떠올려야 하나

앞으로 아래 신호가 보이면 바로 이 문제를 떠올리면 돼.

### 신호 1

**여러 번 거래 가능**

→ 한 번의 매수/매도로 끝내는 문제가 아닐 수 있음

### 신호 2

**동시에 여러 주식은 못 듦**

→ 매수/매도 순서만 지키면 됨
→ 거래를 쪼개는 것이 가능함

### 신호 3

**수수료 없음 / 쿨다운 없음 / 거래 횟수 제한 없음**

→ 오르는 구간을 전부 챙겨도 손해가 없음
→ 그리디 가능성 매우 큼

### 신호 4

**가격 배열이 시간 순서대로 주어짐**

→ 정렬해서 풀면 안 되고, 인접 관계가 중요함

이 신호들을 보면 바로:

> “아, 오르는 날의 차이를 다 더하는 문제구나”

를 떠올리면 돼.

---

## 7. 파이썬 포인트

이 문제에서 중요한 파이썬 포인트는 많지 않아.
자료구조 문제라기보다 **배열 순회** 문제라서 그래.

### 꼭 필요한 것

* `range(1, len(prices))`
* 인접 원소 비교
* 누적합 변수

예시:

```python
from typing import List

class Solution:
    def maxProfit(self, prices: List[int]) -> int:
        profit = 0

        for i in range(1, len(prices)):
            if prices[i] > prices[i - 1]:
                profit += prices[i] - prices[i - 1]

        return profit
```

### 있으면 깔끔한 것

`sum` + generator

```python
class Solution:
    def maxProfit(self, prices: List[int]) -> int:
        return sum(
            max(0, prices[i] - prices[i - 1])
            for i in range(1, len(prices))
        )
```

### 안 필요한 것

이 문제에선 거의 필요 없음:

* `deque`
* `set`
* `defaultdict`
* `sort key`

`enumerate`도 쓸 수는 있지만 필수는 아니야.

---

## 8. 네 접근이 왜 막혔는지 한 문장으로 요약

> **“언제 한 번 크게 먹을까?”를 찾으려 해서 막혔고,
> 사실은 “오를 때마다 조금씩 다 먹는다”가 정답이었다.**

---

## 9. 앞으로 같은 류에서 떠올릴 문장

이 문장을 머리에 박아두면 좋아.

> **거래 제한 없고 수수료/쿨다운 없으면, 양수인 인접 차이를 전부 더하자.**

이게 거의 트리거 문장이다.

---

## 10. 네 메모를 더 좋은 형태로 바꾸면

네 사고 흔적을 문제 풀이용 언어로 바꾸면 이렇게 정리할 수 있어:

* 최솟값/최댓값을 찾는 문제가 아니다.
* 상승 구간이 여러 번 나오면 거래도 여러 번 해야 한다.
* 전체 최대 이익은 각 상승분의 합과 같다.
* 따라서 `prices[i] > prices[i-1]`일 때만 차이를 더하는 그리디를 쓴다.

---

## 11. 이 문제에서 네가 얻어가야 할 진짜 교훈

이 문제의 교훈은 주식 문제가 아니라 이것이야:

> **최적화 문제라고 해서 항상 “최적의 한 지점”을 찾아야 하는 건 아니다.**
> 어떤 문제는 **좋은 국소 선택들을 전부 합치면 전역 최적**이 된다.

이게 그리디 감각이야.

원하면 다음 답변에서 네 고민 흔적을 기준으로
`왜 Stock I는 최솟값 추적이고, Stock II는 양수 차이 합인지`까지 비교해서 정리해줄게.

출처: From training data.
